In [2]:
import pandas as pd
import shap

from eda_package.data import DataManager
from eda_package.features import FeatureEngineer
from eda_package.preprocessor import PreprocessorManager
from eda_package.model import ModelManager
from eda_package.explainer import *

In [3]:
data_manager = DataManager()
feature_engineer = FeatureEngineer()
preprocessor_manager = PreprocessorManager()
model_manager = ModelManager()
explainer_manager = ExplainerManager()

preprocessor_manager.load()
model_manager.load()

X_train, X_test, y_train, y_test = data_manager.prepare_train_test_data()

In [4]:
X_train_fe = feature_engineer.engineer_features(X_train.copy())
X_test_fe = feature_engineer.engineer_features(X_test.copy())

X_train_processed = preprocessor_manager.transform(X_train_fe)
X_test_processed = preprocessor_manager.transform(X_test_fe)

feature_names = preprocessor_manager.preprocessor.get_feature_names_out()

X_train_shap = explainer_manager.transform_to_shap_df(
    X_processed=X_train_processed,
    feature_names=feature_names,
    index=X_train.index,
)

X_test_shap = explainer_manager.transform_to_shap_df(
    X_processed=X_test_processed,
    feature_names=feature_names,
    index=X_test.index,
)

KeyError: "['country', 'reservation_status'] not in index"

In [ ]:
background = X_train_shap.iloc[:50]
explainer_manager.build_explainer(model_manager, background)

In [ ]:
local_result = explainer_manager.explain_local(X_test_shap.iloc[:20], row_index=0)

grouped_local = local_result["grouped_local_shap"]
grouped_local.head(10)

In [ ]:
higher_risk, lower_risk = explainer_manager.split_local_drivers(grouped_local, top_n=5)

print("Higher cancellation risk")
display(higher_risk)

print("Lower cancellation risk")
display(lower_risk)

In [ ]:
global_result = explainer_manager.explain_global(X_test_shap.iloc[:20])

grouped_global = global_result["grouped_global_shap"]
grouped_global.head(15)

In [ ]:
X_test_dates = X_test.copy()
X_test_dates["arrival_date"] = pd.to_datetime(
    X_test_dates["arrival_date_day_of_month"].astype(str)
    + " "
    + X_test_dates["arrival_date_month"].astype(str)
    + " "
    + X_test_dates["arrival_date_year"].astype(str),
    format="%d %B %Y",
    errors="coerce"
)

X_test_dates["arrival_date"].value_counts().head(10)

In [ ]:
date_result = explainer_manager.explain_global_for_date(
    selected_date="2017-07-15",
    X_raw=X_test,
    feature_engineer=feature_engineer,
    preprocessor_manager=preprocessor_manager,
    min_rows=5,
)

print(date_result["n_bookings"])
print(date_result["message"])

if date_result["grouped_global_shap"] is not None:
    display(date_result["grouped_global_shap"].head(15))

In [ ]:
local_shap_values = local_result["shap_values"]
shap.plots.waterfall(local_shap_values[0], max_display=15)

In [ ]:
global_shap_values = global_result["shap_values"]
shap.plots.beeswarm(global_shap_values, max_display=15)